# 070 CASE 070 — Commercial Marine: Öresund

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/imintengine/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_070-Commercial-Marine-Oresund.ipynb)

**Purpose:** Demonstrate tiered vessel-detection combining (1) open Sentinel-2 screening, (2) commercial VHR (Maxar) for detail, and (3) on-orbit AI (PandionAI). Complements [070_CASE_030](070_CASE_030-Marine-Vessels-Sotenas.ipynb); this case is explicitly about the commercial data chain.

**Partners:**
- **Maxar International Sweden** — WorldView-2/3 VHR
- **PandionAI** — AlertSat onboard AI
- **SSC** — ground stations and tasking
- **AI Sweden** — Edge Lab

**Licence:** CC0 1.0 notebook. YOLO11s **AGPL-3.0** (see notice below). Maxar data commercial.

## ⚠️ YOLO11s licence note

The `marine_vessels` analyzer uses Ultralytics YOLO11s which is **AGPL-3.0**.

| Use | Needs Enterprise licence? |
|-----|---------------------------|
| This notebook on Binder / open JupyterHub | No — AGPL network clause satisfied |
| Local research/education | No |
| Commercial closed-source product | **Yes** — [Ultralytics Enterprise](https://www.ultralytics.com/license) |

AGPL-free alternatives in ImintEngine: `ai2_vessels` (AI2 model, different licence) and `object_detection` in heatmap mode (no model).

## 1. Method — tiered detection

```
Step 1: Sentinel-2 (10 m, free)
       → NDWI-filter + YOLO screening
       → density heatmap
       → hotspots for tier-2 tasking

Step 2: Maxar VHR (0.3 m, licensed)
       → detail analysis on hotspots
       → class + length + heading

Step 3: PandionAI onboard (realtime)
       → change detection in orbit
       → downlink anomalies only → saves bandwidth
```

`marine_vessels` outputs: `regions` (bbox dicts), `vessel_count`, `vessel_density_per_km2`.

## 2. Setup

In [ ]:
from executors.local import LocalExecutor
from imint.engine import run_job
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import folium

def get_outputs(result, name):
    for ar in result.analyzer_results:
        if ar.analyzer == name and ar.success:
            return ar.outputs
    return None

# Öresund — dense commercial shipping
AOI = {"west": 12.65, "south": 55.60, "east": 12.85, "north": 55.80}
DATE = "2024-07-20"
print(f"AOI (Öresund): {AOI}")
print(f"Date: {DATE}")

## 3. Tier 1 — Sentinel-2 screening

In [ ]:
executor = LocalExecutor(output_dir="outputs/marine_commercial")
job = executor.build_job(date=DATE, coords=AOI)
result = run_job(job)

mv = get_outputs(result, "marine_vessels")
if mv is not None:
    print(f"YOLO11s: {mv['vessel_count']} vessels")
    print(f"Density: {mv['vessel_density_per_km2']} per km²")

ai2 = get_outputs(result, "ai2_vessels")
if ai2 is not None:
    print(f"AI2: {ai2.get('vessel_count', 'n/a')} vessels")

od = get_outputs(result, "object_detection")
if od is not None:
    print(f"Anomaly regions (variance heatmap): {len(od.get('regions', []))}")

## 4. Visualize

In [ ]:
center = [(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2]
m = folium.Map(location=center, zoom_start=11, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satellite",
).add_to(m)
folium.Rectangle(
    bounds=[[AOI["south"], AOI["west"]], [AOI["north"], AOI["east"]]],
    color="#005cc5", weight=2, fill=False, popup="Öresund AOI",
).add_to(m)
folium.LayerControl().add_to(m)
m

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(job.rgb)
axes[0].set_title(f"Sentinel-2 RGB + detections ({DATE})")
axes[0].axis("off")

if mv is not None:
    for r in mv.get("regions", []):
        bb = r["bbox"]
        axes[0].add_patch(mpatches.Rectangle(
            (bb["x_min"], bb["y_min"]),
            bb["x_max"] - bb["x_min"],
            bb["y_max"] - bb["y_min"],
            linewidth=1.5, edgecolor="red", facecolor="none",
        ))

if od is not None and "heatmap" in od:
    axes[1].imshow(od["heatmap"], cmap="hot")
    axes[1].set_title("Variance heatmap — anomaly hotspots")
else:
    axes[1].text(0.5, 0.5, "object_detection unavailable", ha="center", va="center")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 5. Interpretation — commercial data chain

| Tier | Cost | Resolution | Coverage | Role |
|------|------|------------|----------|------|
| Sentinel-2 | 0 SEK | 10 m | Global, 5 days | Screening, trend |
| Maxar WV-2/3 | ~40 USD/km² | 0.3–1.2 m | On-demand | Detail, evidence |
| PandionAI onboard | Subscription | Variable | Realtime | Alert, event detection |

**SDL 3.0 linkage:**
- Maxar: 3–4 master-thesis projects (NeRF, GAN, sensor fusion)
- PandionAI: time contribution, thesis funding, value-added data
- SSC: ground stations + commercial services
- Combined: shows how public infrastructure (Copernicus, DES) scales via commercial tiers.

### Links
- [Maxar](https://www.maxar.com)
- [PandionAI AlertSat](https://www.pandionai.com)
- [SSC Ground Stations](https://sscspace.com)